# Cylinder VTK -> MeshGraphNet dataset

OpenFOAM が出力した `internal.vtu` を読み込み、2D 平面グラフへ変換して `.pt` に保存します。

## 必要ライブラリと最小設定

`pyvista` でVTKを読み、NumPyでメッシュを扱い、PyTorch Geometricの `Data` に詰めます。

In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv
import torch
from torch_geometric.data import Data

DTYPE = torch.float32
torch.manual_seed(0)

print(torch.__version__, pv.__version__)


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.12.0a0+0291f960b6.nv26.04.48445190 0.48.4


## OpenFOAMの結果を読み込む

In [ ]:
input_dir = Path("data/raw/VTK_000")
output_dir = Path("data/preprocessed")
file_pattern = "case_dir_*/internal.vtu"

files = sorted(input_dir.glob(file_pattern))

print(len(files))  # 501

def read_vtk(path: Path) -> pv.UnstructuredGrid:
    mesh = pv.read(path)
    if not isinstance(mesh, pv.UnstructuredGrid):
        mesh = mesh.cast_to_unstructured_grid()
    return mesh

first_mesh = read_vtk(files[0])  # ひとつめ(t=0)のスナップショット
print(first_mesh.n_points, first_mesh.n_cells)  # 頂点総数, セル総数


## 座標・接続・物理量を抽出する

In [ ]:
def extract_snapshot(mesh: pv.UnstructuredGrid):
    """スナップショットから頂点の座標、物理場、辺を抽出して返す"""
    z = mesh.points[:, 2]
    plane_ids = np.flatnonzero(np.isclose(z, z.min()))  # z下側面の頂点IDだけ取得
    pos = np.asarray(mesh.points[plane_ids, :2], dtype=np.float32)
    source_to_node = np.full(mesh.n_points, -1, dtype=np.int64)
    source_to_node[plane_ids] = np.arange(len(plane_ids))

    edges = set()
    for cell_index in range(mesh.n_cells):
        ids = source_to_node[np.asarray(mesh.get_cell(cell_index).point_ids)]
        ids = np.unique(ids[ids >= 0])
        if len(ids) < 3:
            continue
        center = pos[ids].mean(axis=0)
        order = ids[np.argsort(np.arctan2(
            pos[ids, 1] - center[1],
            pos[ids, 0] - center[0],
        ))]
        edges.update(
            (min(a, b), max(a, b))
            for a, b in zip(order, np.roll(order, -1))
        )

    target_fields = ("U", "p")
    undirected = np.asarray(sorted(edges), dtype=np.int64)
    edge_index = np.concatenate((undirected, undirected[:, ::-1]), axis=0).T  # 双方向を考慮
    velocity = np.asarray(mesh.point_data[target_fields[0]])[plane_ids, :2]
    pressure = np.asarray(mesh.point_data[target_fields[1]])[plane_ids].reshape(-1, 1)
    state = np.concatenate((velocity, pressure), axis=1).astype(np.float32)
    return pos, edge_index, state  # 頂点座標、辺、物理場

pos, edge_index, state = extract_snapshot(first_mesh)
print(pos.shape)  # (4984, 2)
print(edge_index.shape)  # (2, 19104)
print(state.shape)  # (4984, 3)

## `pos`, `edge_index`, `state` を `Data` に変換する

In [ ]:
pos_tensor = torch.tensor(pos, dtype=DTYPE)
edge_index_tensor = torch.tensor(edge_index, dtype=torch.long)
x_tensor = torch.tensor(state, dtype=DTYPE)

def to_data(current_state: torch.Tensor, next_state: torch.Tensor) -> Data:
    return Data(
        pos=pos_tensor,  # ノード座標
        x=current_state,  # 現在時刻のノード特徴量
        edge_index=edge_index_tensor,  # エッジ接続
        y=next_state,  # 次の時刻のノード特徴量
    )

states = [extract_snapshot(read_vtk(path))[2] for path in files]
dataset = [
    to_data(current, next)
    for current, next in zip(states, states[1:])
]
print(len(dataset))

## `.pt`形式で保存する

In [ ]:
output_path = output_dir / "cylinder_dataset.pt"
torch.save(dataset, output_path)

loaded = torch.load(output_path, weights_only=False)
first = loaded[0]
print(len(loaded))
print(first.pos.shape)
print(first.x.shape)
print(first.y.shape)
print(first.edge_index.shape)